<a href="https://colab.research.google.com/github/vtecftwy/utseus-dives/blob/main/nbs/genai_3.3_agent_w_langchain.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Build a multi task agent with Langchain

Inspired by LangChain supervisor agent tutorial [here](https://docs.langchain.com/oss/python/langchain/supervisor)

<p>In this notebook we will use the following python packages on Python 3.10.15:</p>
<ul>
<li><code>eccore</code></li>
<li><code>jupyter</code></li>
<li><code>langchain</code></li>
<li><code>matplotlib</code></li>
<li><code>numpy</code></li>
<li><code>pillow</code></li>
<li><code>python-dotenv</code></li>
</ul>

<details>
    <summary><i>List of Python packages with versions:<i></summary>
<p></br>The notebook was last tested with the following versions:</p>
<ul>
<li><code>eccore==0.0.5</code></li>
<li><code>jupyter==1.1.1</code></li>
<li><code>langchain==1.1.3</code></li>
<li><code>matplotlib==3.10.7</code></li>
<li><code>numpy==2.3.3</code></li>
<li><code>pillow==11.3.0</code></li>
<li><code>python-dotenv==1.1.1</code></li>
</ul>
</details>     

## 0. Installs, Imports, Utility Fctn and Setup

In [1]:
import os
import textwrap
from dotenv import load_dotenv
from eccore.comms import clean_no_proxy_12vpn
from langchain_community.docstore.in_memory import InMemoryDocstore
from langchain_community.vectorstores import FAISS
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_openai import OpenAIEmbeddings
from langchain.tools import tool
from pathlib import Path
from pprint import pprint

clean_no_proxy_12vpn()

'100.64.*,172.16.*,192.168.*,10.*,127.*,169.254.*,localhost,*.local,vpnguistub'

In [2]:
def show_attr(o):
    for a in o.__dict__:
        if not '_' in a:
            print(f"{a:10s}: \t{getattr(o, a)}")

def printmd(*txt):
    for t in txt:
        display(Markdown(t))

In [3]:
load_dotenv()
api_key = os.getenv("OPENAI_API_KEY")

## Define model

In [4]:
from langchain.chat_models import init_chat_model

model = init_chat_model("gpt-4.1-mini")

## Define tools

In [5]:
@tool
def send_email(
    to: list[str],  # email addresses
    subject: str,
    body: str,
    cc: list[str] = []
) -> str:
    """Send an email via email API. Requires properly formatted addresses."""
    # Stub: In practice, this would call SendGrid, Gmail API, etc.
    return f"Email sent to {', '.join(to)} - Subject: {subject}\n{body[:100]}"


In [6]:
@tool
def create_calendar_event(
    title: str,
    start_time: str,       # ISO format: "2024-01-15T14:00:00"
    end_time: str,         # ISO format: "2024-01-15T15:00:00"
    attendees: list[str],  # email addresses
    location: str = ""
) -> str:
    """Create a calendar event. Requires exact ISO datetime format."""
    # Stub: In practice, this would call Google Calendar API, Outlook API, etc.
    return f"Event created: {title} from {start_time} to {end_time} with {len(attendees)} attendees"

In [7]:
@tool
def get_available_time_slots(
    attendees: list[str],
    date: str,  # ISO format: "2024-01-15"
    duration_minutes: int
) -> list[str]:
    """Check calendar availability for given attendees on a specific date."""
    # Stub: In practice, this would query calendar APIs
    # print('will return fixed time slots for demo purposes: 09:00, 14:00, 16:00')
    return ["09:00", "14:00", "16:00"]

## Create specialized sub-agents

### Create a calendar agent

The calendar agent:
- understands requests in natural language to schedule a meeting, and
- translates them into precise API calls. 

It handles date parsing, availability checking, and event creation.

In [8]:
from langchain.agents import create_agent

CALENDAR_AGENT_PROMPT = (
    "You are a calendar scheduling assistant. "
    "Parse natural language scheduling requests (e.g., 'next Tuesday at 2pm') "
    "into proper ISO datetime formats. "
    "Use get_available_time_slots to check availability when needed. "
    "Use create_calendar_event to schedule events. "
    "Always confirm what was scheduled in your final response."
)

calendar_agent = create_agent(
    model,
    tools=[create_calendar_event, get_available_time_slots],
    system_prompt=CALENDAR_AGENT_PROMPT,
)

Test the calendar agent

In [9]:
query = "Schedule a meeting with John Doe next Tuesday at 2pm for 1 hour"

for step in calendar_agent.stream(
    {"messages": [{"role": "user", "content": query}]}
):
    for update in step.values():
        for message in update.get("messages", []):
            message.pretty_print()

================================== Ai Message ==================================
Tool Calls:
  get_available_time_slots (call_bUmty4OJf1UKwEPsfsL46zSK)
 Call ID: call_bUmty4OJf1UKwEPsfsL46zSK
  Args:
    attendees: ['John Doe']
    date: 2024-06-18
    duration_minutes: 60
================================= Tool Message =================================
Name: get_available_time_slots

["09:00", "14:00", "16:00"]
================================== Ai Message ==================================
Tool Calls:
  create_calendar_event (call_nf4rIyABEiX2CxRd7fiohI0I)
 Call ID: call_nf4rIyABEiX2CxRd7fiohI0I
  Args:
    title: Meeting with John Doe
    start_time: 2024-06-18T14:00:00
    end_time: 2024-06-18T15:00:00
    attendees: ['John Doe']
    location:
================================= Tool Message =================================
Name: create_calendar_event

Event created: Meeting with John Doe from 2024-06-18T14:00:00 to 2024-06-18T15:00:00 with 1 attendees
===============================

### Create an email agent

In [10]:
create_agent?

Signature:
create_agent(
    model: 'str | BaseChatModel',
    tools: 'Sequence[BaseTool | Callable | dict[str, Any]] | None' = None,
    *,
    system_prompt: 'str | SystemMessage | None' = None,
    middleware: 'Sequence[AgentMiddleware[StateT_co, ContextT]]' = (),
    response_format: 'ResponseFormat[ResponseT] | type[ResponseT] | None' = None,
    state_schema: 'type[AgentState[ResponseT]] | None' = None,
    context_schema: 'type[ContextT] | None' = None,
    checkpointer: 'Checkpointer | None' = None,
    store: 'BaseStore | None' = None,
    interrupt_before: 'list[str] | None' = None,
    interrupt_after: 'list[str] | None' = None,
    debug: 'bool' = False,
    name: 'str | None' = None,
    cache: 'BaseCache | None' = None,
) -> 'CompiledStateGraph[AgentState[ResponseT], ContextT, _InputAgentState, _OutputAgentState[ResponseT]]'
Docstring:
Creates an agent graph that calls tools in a loop until a stopping condition is met.

For more details on using `create_agent`,
visit the 

In [11]:
EMAIL_AGENT_PROMPT = (
    "You are an email assistant. "
    "Compose professional emails based on natural language requests. "
    "Extract recipient information and craft appropriate subject lines and body text. "
    "Use send_email to send the message. "
    "Always confirm what was sent in your final response."
    "I am Jean Dupont, Head of Operations at Acme Corp."
    
)

email_agent = create_agent(
    model,
    tools=[send_email],
    system_prompt=EMAIL_AGENT_PROMPT,
)

Test the email agent

In [12]:
query = "Send the design team a reminder about reviewing the new mockups"

for step in email_agent.stream(
    {"messages": [{"role": "user", "content": query}]}
):
    for update in step.values():
        for message in update.get("messages", []):
            message.pretty_print()

================================== Ai Message ==================================
Tool Calls:
  send_email (call_wHS1uvHEG9YlxfM4XIAsb5Rk)
 Call ID: call_wHS1uvHEG9YlxfM4XIAsb5Rk
  Args:
    to: ['design_team@acmecorp.com']
    subject: Reminder: Review of New Mockups
    body: Dear Design Team,

This is a friendly reminder to review the new mockups at your earliest convenience. Your feedback is important to ensure we move forward effectively.

Thank you for your attention to this matter.

Best regards,
Jean Dupont
Head of Operations
Acme Corp.
================================= Tool Message =================================
Name: send_email

Email sent to design_team@acmecorp.com - Subject: Reminder: Review of New Mockups
Dear Design Team,

This is a friendly reminder to review the new mockups at your earliest convenienc
================================== Ai Message ==================================

I have sent a reminder email to the design team about reviewing the new mockups. If yo

## Wrap sub-agents as tools

Now wrap each sub-agent as a tool that the supervisor can invoke. This is the key architectural step that creates the layered system. The supervisor will see high-level tools like “schedule_event”, not low-level tools like “create_calendar_event”.

In [13]:
@tool
def schedule_event(request: str) -> str:
    """Schedule calendar events using natural language.

    Use this when the user wants to create, modify, or check calendar appointments.
    Handles date/time parsing, availability checking, and event creation.

    Input: Natural language scheduling request (e.g., 'meeting with design team
    next Tuesday at 2pm')
    """
    result = calendar_agent.invoke({
        "messages": [{"role": "user", "content": request}]
    })
    return result["messages"][-1].text


@tool
def manage_email(request: str) -> str:
    """Send emails using natural language.

    Use this when the user wants to send notifications, reminders, or any email
    communication. Handles recipient extraction, subject generation, and email
    composition.

    Input: Natural language email request (e.g., 'send them a reminder about
    the meeting')
    """
    result = email_agent.invoke({
        "messages": [{"role": "user", "content": request}]
    })
    return result["messages"][-1].text

##  Create the supervisor agent

In [14]:
SUPERVISOR_PROMPT = (
    "You are a helpful personal assistant. "
    "You can schedule calendar events and send emails. "
    "Break down user requests into appropriate tool calls and coordinate the results. "
    "When a request involves multiple actions, use multiple tools in sequence."
)

supervisor_agent = create_agent(
    model,
    tools=[schedule_event, manage_email],
    system_prompt=SUPERVISOR_PROMPT,
)

## Use the supervisor

### Simple single-domain request

In [15]:
query = "Schedule a team standup for tomorrow at 9am"

for step in supervisor_agent.stream(
    {"messages": [{"role": "user", "content": query}]}
):
    for update in step.values():
        for message in update.get("messages", []):
            message.pretty_print()

================================== Ai Message ==================================
Tool Calls:
  schedule_event (call_5pwOpTh8r1KWsvUSlbLZD77t)
 Call ID: call_5pwOpTh8r1KWsvUSlbLZD77t
  Args:
    request: team standup tomorrow at 9am
================================= Tool Message =================================
Name: schedule_event

The team standup is scheduled for tomorrow, June 10th, from 9:00 AM to 9:30 AM. Let me know if you need anything else!
================================== Ai Message ==================================

The team standup is scheduled for tomorrow at 9:00 AM to 9:30 AM. Is there anything else you would like to do?


### Complex multi-domain request

In [16]:
query = (
    "Schedule a meeting with the design team next Tuesday at 2pm for 1 hour, "
    "and send them an email reminder about reviewing the new mockups."
)

for step in supervisor_agent.stream(
    {"messages": [{"role": "user", "content": query}]}
):
    for update in step.values():
        for message in update.get("messages", []):
            message.pretty_print()

================================== Ai Message ==================================
Tool Calls:
  schedule_event (call_xnSfCSXWI6a4oYqSTkbWAsTP)
 Call ID: call_xnSfCSXWI6a4oYqSTkbWAsTP
  Args:
    request: meeting with design team next Tuesday at 2pm for 1 hour
  manage_email (call_KUMqBLREibVYwpONMrwjxLpy)
 Call ID: call_KUMqBLREibVYwpONMrwjxLpy
  Args:
    request: send an email reminder to the design team about reviewing the new mockups
================================= Tool Message =================================
Name: schedule_event

I have scheduled the meeting with the design team on next Tuesday, May 28th, from 2:00 PM to 3:00 PM.
================================= Tool Message =================================
Name: manage_email

I have sent an email reminder to the design team about reviewing the new mockups. Is there anything else you would like to do?
================================== Ai Message ==================================

The meeting with the design team is schedule

### Complex multi-domain request 2

In [17]:
query = (
    "Schedule a meeting with the design team next Tuesday at 2pm for 1 hour, "
    "and send them an email reminder about reviewing the new mockups."
    "Then schedule a meeting with John next Friday at 3 pm for 30 minutes. If not available, pick the next available slot. "
    "And send a email to John about the meeting, telling him that he has to prepare a presentation for Project x."
)

for step in supervisor_agent.stream(
    {"messages": [{"role": "user", "content": query}]}
):
    for update in step.values():
        for message in update.get("messages", []):
            message.pretty_print()

================================== Ai Message ==================================
Tool Calls:
  schedule_event (call_VNHAoK0yveEPHIrO5aUSrHfL)
 Call ID: call_VNHAoK0yveEPHIrO5aUSrHfL
  Args:
    request: meeting with the design team next Tuesday at 2pm for 1 hour
  schedule_event (call_rTlaJAYFgGmqwORc7J6e4Myn)
 Call ID: call_rTlaJAYFgGmqwORc7J6e4Myn
  Args:
    request: meeting with John next Friday at 3 pm for 30 minutes. If not available, pick the next available slot.
================================= Tool Message =================================
Name: schedule_event

I have scheduled the meeting with the design team on next Tuesday, May 14th, from 2:00 PM to 3:00 PM.
================================= Tool Message =================================
Name: schedule_event

John is not available at 3 pm next Friday, so I scheduled the meeting at the next available slot, which is 2 pm on May 31, 2024. The meeting will last for 30 minutes.
================================== Ai Message ====